In [11]:

# 1. Get the path to the project root relative to this file
import os
import sys
# This points to: d:\pyhton_projects\Test_AI_LLM_Apps
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))

# 2. Add that root folder to the path
if project_root not in sys.path:
    sys.path.insert(0, project_root) # insert at 0 to prioritize this path
from langchain.messages import HumanMessage
from deepeval.test_case import LLMTestCase, llm_test_case
from deepeval.dataset import EvaluationDataset
from modules.langchain_agent import agent
from notebooks.RAG.src.RagRetriever import RagRetriever



In [12]:

rag_agent = agent()
llmTest = LLMTestCase(
    input="What is Gautam skill set?",
    actual_output=rag_agent.invoke({
    "messages": [HumanMessage(content="What is Gautam skill set?")]})["messages"][-1].content,
    expected_output="Java, Python, Langchain"
    )


data_set = EvaluationDataset()
data_set.add_test_case(llmTest)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 853.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
llmTest

LLMTestCase(input='What is Gautam skill set?', actual_output="Gautam's skill set includes Artificial Intelligence and Machine Learning.", expected_output='Java, Python, Langchain', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None)

In [14]:
from notebooks.RAG.src.EmbeddingManager import EmbeddingManager
from notebooks.RAG.src.VectorStoreManager import VectorStoreManager


def get_retrieval_context(query: str) -> list[str]:
    """Return just the retrieved documents as a list of strings."""
    persist_directory = os.path.join(project_root, "notebooks", "RAG", "store")
    vector_store_manager = VectorStoreManager(persist_directory=persist_directory)
    embeddings_manager = EmbeddingManager()
    rag_retriever = RagRetriever(vector_store_manager, embeddings_manager)

    retrieval_result = rag_retriever.retrieve(query, top_k=3)
    # Flatten nested lists into a simple list of strings
    return [doc for docs in retrieval_result.get("documents", []) for doc in docs]


def prompt_with_context(query: str) -> str:
    """Return a formatted prompt string for the agent."""
    docs_content = get_retrieval_context(query)
    return (
        "You are an assistant for question-answering tasks. "
        "Use the following retrieved context to answer. "
        f"\n\nContext:\n{docs_content}"
    )

In [15]:
query = "List Down total experience of Gautam from all companies? He had worked in dfferent companies just provide all company names."

# Build message for agent
test_message = HumanMessage(content=query)

# Invoke agent
response = rag_agent.invoke({"messages": [test_message]})

# Extract retrieval context for test case
retrieval_context = get_retrieval_context(query)

# Build test case
llmTest = LLMTestCase(
    input=query,
    actual_output=response["messages"][-1].content,
    expected_output="Cognizant, TIAA, FIS,Xavient",
    retrieval_context=retrieval_context   # ✅ list[str]
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3262.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3408.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
llmTest

LLMTestCase(input='List Down total experience of Gautam from all companies? He had worked in dfferent companies just provide all company names.', actual_output='The total experience of Gautam is 12 years and 6 months. He had worked in different companies such as TIAA GBS, University, and others.', expected_output='Cognizant, TIAA, FIS,Xavient', context=None, retrieval_context=['SkillsWORK EXPERIENCE\nTIAA GBS2021- Current\nAssociate Specialist', 'University (2011)\n❑ Contributed as a QA Manual and Automation Engineer on a', 'workflows.\nGAUTAM RAWAT Contact - +91 – 8939725506\nSDET Email – gautam.rawat123@gmail.com'], additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None)

In [17]:
from deepeval.metrics import ContextualPrecisionMetric,ContextualRelevancyMetric
from modules.local_model_llama import get_local_model
local_judge=get_local_model()

contextualRelevancyMetric=ContextualRelevancyMetric(threshold=.5,model=local_judge)
contextualPrecisionMetric=ContextualPrecisionMetric(threshold=0.7,model=local_judge)

In [20]:
from deepeval import evaluate


evaluate(test_cases=[llmTest],metrics=[contextualRelevancyMetric])

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using Llama-3.2-3B-Instruct-Q4_K_M, strict=False,
async_mode=True)...



Metrics Summary

  - ❌ Contextual Relevancy (score: 0.2, threshold: 0.5, strict: False, evaluation model: Llama-3.2-3B-Instruct-Q4_K_M, reason: The score is 0.20 because the statement does not provide any information about Gautam's total experience across all companies. It only mentions his current position at TIAA GBS., error: None)

For test case:

  - input: List Down total experience of Gautam from all companies? He had worked in dfferent companies just provide all company names.
  - actual output: The total experience of Gautam is 12 years and 6 months. He had worked in different companies such as TIAA GBS, University, and others.
  - expected output: Cognizant, TIAA, FIS,Xavient
  - context: None
  - retrieval context: ['SkillsWORK EXPERIENCE\nTIAA GBS2021- Current\nAssociate Specialist', 'University (2011)\n❑ Contributed as a QA Manual and Automation Engineer on a', 'workflows.\nGAUTAM RAWAT Contact - +91 – 8939725506\nSDET Email – gautam.rawat123@gmail.com']


Overall Metric 

⚠ WARNING: No hyperparameters logged.
» ]8;id=471638;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 69.9s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Contextual Relevancy', threshold=0.5, success=False, score=0.2, reason="The score is 0.20 because the statement does not provide any information about Gautam's total experience across all companies. It only mentions his current position at TIAA GBS.", strict_mode=False, evaluation_model='Llama-3.2-3B-Instruct-Q4_K_M', error=None, evaluation_cost=None, verbose_logs='Verdicts:\n[\n    {\n        "verdicts": [\n            {\n                "statement": "SkillsWORK EXPERIENCE\\nTIAA GBS2021- Current\\nAssociate Specialist",\n                "verdict": "no",\n                "reason": "The statement does not provide any information about Gautam\'s total experience across all companies. It only mentions his current position at TIAA GBS."\n            }\n        ]\n    },\n    {\n        "verdicts": [\n            {\n                "statement": "University (2011)",\n                "

In [21]:

evaluate(test_cases=[llmTest],metrics=[contextualPrecisionMetric])

✨ You're running DeepEval's latest Contextual Precision Metric! (using Llama-3.2-3B-Instruct-Q4_K_M, strict=False,
async_mode=True)...

d:\pyhton_projects\Test_AI_LLM_Apps\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Contextual Precision (score: 1.0, threshold: 0.7, strict: False, evaluation model: Llama-3.2-3B-Instruct-Q4_K_M, reason: The score is 1.00 because the relevant nodes are ranked higher than irrelevant nodes. The first two nodes provide the total experience of Gautam from all companies, while the third node does not., error: None)

For test case:

  - input: List Down total experience of Gautam from all companies? He had worked in dfferent companies just provide all company names.
  - actual output: The total experience of Gautam is 12 years and 6 months. He had worked in different companies such as TIAA GBS, University, and others.
  - expected output: Cognizant, TIAA, FIS,Xavient
  - context: None
  - retrieval context: ['SkillsWORK EXPERIENCE\nTIAA GBS2021- Current\nAssociate Specialist', 'University (2011)\n❑ Contributed as a QA Manual and Automation Engineer on a', 'workflows.\nGAUTAM RAWAT Contact - +91 – 8939725506\nSDET Email – gautam.rawat123@gmail.com']

⚠ WARNING: No hyperparameters logged.
» ]8;id=34944;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 38.41s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Contextual Precision', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because the relevant nodes are ranked higher than irrelevant nodes. The first two nodes provide the total experience of Gautam from all companies, while the third node does not.', strict_mode=False, evaluation_model='Llama-3.2-3B-Instruct-Q4_K_M', error=None, evaluation_cost=None, verbose_logs='Verdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": "The context mentions \'SkillsWORK EXPERIENCE\' which indicates work experience across different companies."\n    },\n    {\n        "verdict": "yes",\n        "reason": "The context provides the company names: TIAA, GBS, University."\n    },\n    {\n        "verdict": "no",\n        "reason": "The context does not provide the total experience of Gautam from all companies."\n    }\n]')], conversational=False, multimodal=False, input='List